# **Extension of the Network Construction Step**

# Import Libraries and Configurations

In [1]:
import json
import os
import sys
from urllib.parse import urljoin

import pandas as pd
import requests

# Add project root to Python's path
sys.path.append(os.path.abspath(os.path.join('..')))

from config import (
    DATA_SUBDIRS,
    NETWORK_DATA_DIRS,
    NETWORK_FILES,
)

GROUPS = DATA_SUBDIRS.copy()

# g:Profiler (GP) API endpoints of interest
GP_API_BASE_URL = 'https://biit.cs.ut.ee/gprofiler/api/'
GP_API_ENDPOINTS = {
    'data_versions': urljoin(GP_API_BASE_URL, 'util/data_versions'),
    'functional_profiling': urljoin(GP_API_BASE_URL, 'gost/profile'),
}

ORGANISM = 'hsapiens'

# Pathway Enrichment

## Functions

In [2]:
def perform_pathway_enrichment(mrnas_df):
    # List all messenger RNAs in the network
    query = mrnas_df['id'].to_list()

    # Define the parameters for functional enrichment analysis
    parameters = {
        'organism': ORGANISM,
        'query': query,
        'sources': ['KEGG'],
        'user_threshold': 0.05,
        'all_results': False,
        'significance_threshold_method': 'fdr',
    }

    # Request analysis for the g:Profile API
    response = requests.post(
        url=GP_API_ENDPOINTS['functional_profiling'],
        json=parameters,
    )
    response.raise_for_status()

    # Convert the response content to a JSON
    json_response = json.loads(response.content.decode('utf-8'))

    # Transform the JSON response result into a DataFrame
    enrichment_df = pd.json_normalize(json_response['result'])

    # Recover the enriched genes/messenger RNAs
    enrichment_df['genes'] = (
        enrichment_df['intersections']
        .apply(
            lambda col: [
                genes for genes, sources in zip(query, col) if sources
            ]
        )
    )

    # Remove the unnecessary columns
    cols = ['native', 'name', 'p_value', 'genes']
    enrichment_df = enrichment_df[cols]

    return enrichment_df

In [3]:
def perform_pathway_enrichment(mrnas_df):
    # List all unique messenger RNAs in the network
    query = (
        mrnas_df['id']
        .dropna()
        .astype(str)
        .drop_duplicates()
        .to_list()
    )

    # Columns expected in the returned DataFrame
    cols = ['native', 'name', 'p_value', 'genes']

    # Return an empty DataFrame if there are no mRNAs
    if not query:
        return pd.DataFrame(columns=cols)

    # Define the parameters for functional enrichment analysis
    parameters = {
        'organism': ORGANISM,
        'query': query,
        'sources': ['KEGG'],
        'user_threshold': 0.05,
        'all_results': False,
        'significance_threshold_method': 'fdr',
    }

    # Request analysis from the g:Profiler API
    response = requests.post(
        url=GP_API_ENDPOINTS['functional_profiling'],
        json=parameters,
        timeout=60,
    )
    response.raise_for_status()

    # Convert the response to JSON
    json_response = response.json()

    # Return an empty DataFrame when no pathways are enriched
    if not json_response.get('result'):
        return pd.DataFrame(columns=cols)

    # Transform enrichment results into a DataFrame
    enrichment_df = pd.json_normalize(json_response['result'])

    # Recover g:Profiler gene-mapping metadata
    query_metadata = (
        json_response['meta']
        ['genes_metadata']
        ['query']
    )

    def recover_genes(row):
        # Results identify which query generated the enrichment result
        metadata = query_metadata[row['query']]

        # Original ID -> Ensembl gene ID(s)
        mapping = metadata['mapping']

        # Ensembl genes in the exact order used by intersections
        ensgs = metadata['ensgs']

        intersections = row['intersections']

        # This relationship should always hold for a valid response
        if len(ensgs) != len(intersections):
            raise ValueError(
                'g:Profiler returned incompatible lengths for '
                f'ensgs ({len(ensgs)}) and intersections '
                f'({len(intersections)}).'
            )

        # Build:
        # Ensembl ID -> original network ID(s)
        reverse_mapping = {}

        for original_id, mapped_ensgs in mapping.items():
            if not mapped_ensgs:
                continue

            for ensg in mapped_ensgs:
                reverse_mapping.setdefault(ensg, []).append(original_id)

        # Recover the original IDs associated with this pathway
        genes = []
        seen = set()

        for ensg, evidence in zip(ensgs, intersections):
            if not evidence:
                continue

            for original_id in reverse_mapping.get(ensg, []):
                if original_id not in seen:
                    genes.append(original_id)
                    seen.add(original_id)

        return genes

    # Recover original network IDs for each enriched pathway
    enrichment_df['genes'] = enrichment_df.apply(
        recover_genes,
        axis=1,
    )

    # Keep only relevant columns
    return enrichment_df[cols]

In [4]:
def create_gene_pathway_network(group):
    # Define the path to the directory related to the group
    group_dir = (group.lower()).replace(' ', '-')
    dir_path = NETWORK_DATA_DIRS['processed'][group_dir]

    # Create a node DataFrame for the microRNA-mRNA interaction network
    file_name = NETWORK_FILES['interaction-nodes']
    interaction_nodes_df = pd.read_csv(os.path.join(dir_path, file_name))
    
    # Pathway enrichment analysis of messenger RNAs
    mrnas_df = interaction_nodes_df.query('type == "Messenger RNA"')
    enrichment_df = perform_pathway_enrichment(mrnas_df)

    # Create a DataFrame of pathway nodes
    pathways_df = (
        enrichment_df[['native', 'name', 'p_value']]
        .rename(columns={
            'native': 'id',
            'name': 'label',
            'p_value': 'qvalue',
        })
    )
    pathways_df['type'] = 'Pathway'

    # Create a DataFrame of edges between genes and pathways
    edges_df = (
        enrichment_df
        .explode(column='genes', ignore_index=True)
        .dropna(subset=['genes'])
        [['genes', 'native']]
        .rename(columns={
            'genes': 'source',
            'native': 'target',
        })
    )
    
    # Create a DataFrame of enriched gene nodes
    enriched_genes_df = (
        edges_df[['source']]
        .rename(columns={'source': 'id'})
        .drop_duplicates(ignore_index=True)
    )
    enriched_genes_df['label'] = enriched_genes_df['id']
    enriched_genes_df['type'] = 'Messenger RNA'

    # Concatenate enriched gene and pathway nodes in a single DataFrame
    nodes_df = (
        pd.concat(
            objs=[enriched_genes_df, pathways_df],
            ignore_index=True
        )
    )

    # Store the DataFrame of edges
    file_name = 'membership-network-edges.csv'
    file_path = os.path.join(dir_path, file_name)
    edges_df.to_csv(file_path, index=False)

    # Store the DataFrame of nodes
    file_name = 'membership-network-nodes.csv'
    file_path = os.path.join(dir_path, file_name)
    nodes_df.to_csv(file_path, index=False)
    
    return edges_df, nodes_df

## Data Versions Endpoint

In [5]:
# Request the API version and status to the endpoint
response = requests.get(
    url=GP_API_ENDPOINTS['data_versions'],
    json={'organism': ORGANISM}
)

# Print the response content
try:
    parsed = json.loads(response.content.decode('utf-8'))
    print(json.dumps(parsed, indent=3))
except json.JSONDecodeError:
    print('Response is not valid JSON:')
    print(response.content.decode('utf-8'))

{
   "biomart": "Ensembl",
   "biomart_version": "114",
   "display_name": "Human",
   "genebuild": "GRCh38.p14",
   "gprofiler_version": "e114_eg62_p19_27110d83",
   "organism": "hsapiens",
   "sources": {
      "CORUM": {
         "name": "CORUM protein complexes",
         "version": "28.11.2022 Corum 4.1"
      },
      "GO:BP": {
         "name": "biological process",
         "version": "annotations: BioMart\nclasses: releases/2026-01-23"
      },
      "GO:CC": {
         "name": "cellular component",
         "version": "annotations: BioMart\nclasses: releases/2026-01-23"
      },
      "GO:MF": {
         "name": "molecular function",
         "version": "annotations: BioMart\nclasses: releases/2026-01-23"
      },
      "HP": {
         "name": "Human Phenotype Ontology",
         "version": "annotations: 03.2026\nclasses: None"
      },
      "HPA": {
         "name": "Human Protein Atlas",
         "version": "annotations: HPA website: 25-11-06\nclasses: script: 26-01-20"
 

## g:GOSt Endpoint

### Basal-like

In [6]:
# Create the gene-pathway membership network for the group
edges_df, nodes_df = create_gene_pathway_network('Basal-like')

In [7]:
# Print all enriched pathways for the group
if not edges_df.empty and not nodes_df.empty:
    print(
        nodes_df
        .query('type == "Pathway"')
        .reset_index(drop=True)
        [['id', 'label', 'qvalue']]
    )

           id                                      label        qvalue
0  KEGG:05202    Transcriptional misregulation in cancer  1.191417e-07
1  KEGG:04211               Longevity regulating pathway  1.164198e-02
2  KEGG:05200                         Pathways in cancer  1.164198e-02
3  KEGG:04020                  Calcium signaling pathway  2.187057e-02
4  KEGG:01521  EGFR tyrosine kinase inhibitor resistance  2.543428e-02
5  KEGG:01522                       Endocrine resistance  4.219136e-02
6  KEGG:04010                     MAPK signaling pathway  4.219136e-02
7  KEGG:04068                     FoxO signaling pathway  4.219136e-02
8  KEGG:05205                    Proteoglycans in cancer  4.219136e-02


In [8]:
# Print the membership edges ordered by q-value
if not edges_df.empty and not nodes_df.empty:
    print(
        pd.merge(
            left=edges_df,
            right=nodes_df,
            how='inner',
            left_on='target',
            right_on='id'
        )
        [['source', 'target', 'label', 'qvalue']]
        .sort_values(by='qvalue', ascending=True, ignore_index=True)
    )

    source      target                                    label        qvalue
0    CCND2  KEGG:05202  Transcriptional misregulation in cancer  1.191417e-07
1   TGFBR2  KEGG:05202  Transcriptional misregulation in cancer  1.191417e-07
2     CD86  KEGG:05202  Transcriptional misregulation in cancer  1.191417e-07
3     IGF1  KEGG:05202  Transcriptional misregulation in cancer  1.191417e-07
4     NGFR  KEGG:05202  Transcriptional misregulation in cancer  1.191417e-07
..     ...         ...                                      ...           ...
87    FGF1  KEGG:04010                   MAPK signaling pathway  4.219136e-02
88  PRKACB  KEGG:04010                   MAPK signaling pathway  4.219136e-02
89    NGFR  KEGG:04010                   MAPK signaling pathway  4.219136e-02
90   ERBB3  KEGG:04010                   MAPK signaling pathway  4.219136e-02
91    IGF1  KEGG:05205                  Proteoglycans in cancer  4.219136e-02

[92 rows x 4 columns]


### HER2-enriched

In [9]:
# Create the gene-pathway membership network for the group
edges_df, nodes_df = create_gene_pathway_network('HER2-enriched')

In [10]:
# Print all enriched pathways for the group
if not edges_df.empty and not nodes_df.empty:
    print(
        nodes_df
        .query('type == "Pathway"')
        .reset_index(drop=True)
        [['id', 'label', 'qvalue']]
    )

In [11]:
# Print the membership edges ordered by q-value
if not edges_df.empty and not nodes_df.empty:
    print(
        pd.merge(
            left=edges_df,
            right=nodes_df,
            how='inner',
            left_on='target',
            right_on='id'
        )
        [['source', 'target', 'label', 'qvalue']]
        .sort_values(by='qvalue', ascending=True, ignore_index=True)
    )

### Luminal A

In [12]:
# Create the gene-pathway membership network for the group
edges_df, nodes_df = create_gene_pathway_network('Luminal A')

In [13]:
# Print all enriched pathways for the group
if not edges_df.empty and not nodes_df.empty:
    print(
        nodes_df
        .query('type == "Pathway"')
        .reset_index(drop=True)
        [['id', 'label', 'qvalue']]
    )

           id                                    label    qvalue
0  KEGG:04390                  Hippo signaling pathway  0.018969
1  KEGG:04510                           Focal adhesion  0.018969
2  KEGG:04518                       Integrin signaling  0.018969
3  KEGG:04820             Cytoskeleton in muscle cells  0.018969
4  KEGG:05202  Transcriptional misregulation in cancer  0.018969
5  KEGG:04151               PI3K-Akt signaling pathway  0.024587
6  KEGG:05165           Human papillomavirus infection  0.037774


In [14]:
# Print the membership edges ordered by q-value
if not edges_df.empty and not nodes_df.empty:
    print(
        pd.merge(
            left=edges_df,
            right=nodes_df,
            how='inner',
            left_on='target',
            right_on='id'
        )
        [['source', 'target', 'label', 'qvalue']]
        .sort_values(by='qvalue', ascending=True, ignore_index=True)
    )

    source      target                           label    qvalue
0    WWTR1  KEGG:04390         Hippo signaling pathway  0.018969
1    ITGA9  KEGG:04518              Integrin signaling  0.018969
2     FLNC  KEGG:04820    Cytoskeleton in muscle cells  0.018969
3    LAMA2  KEGG:04820    Cytoskeleton in muscle cells  0.018969
4   COL5A1  KEGG:04820    Cytoskeleton in muscle cells  0.018969
..     ...         ...                             ...       ...
62  ITGA11  KEGG:05165  Human papillomavirus infection  0.037774
63   ITGA9  KEGG:05165  Human papillomavirus infection  0.037774
64  PPP2CB  KEGG:05165  Human papillomavirus infection  0.037774
65    HEY2  KEGG:05165  Human papillomavirus infection  0.037774
66  PARD6B  KEGG:05165  Human papillomavirus infection  0.037774

[67 rows x 4 columns]


### Luminal B

In [15]:
# Create the gene-pathway membership network for the group
edges_df, nodes_df = create_gene_pathway_network('Luminal B')

In [16]:
# Print all enriched pathways for the group
if not edges_df.empty and not nodes_df.empty:
    print(
        nodes_df
        .query('type == "Pathway"')
        .reset_index(drop=True)
        [['id', 'label', 'qvalue']]
    )

            id                                              label    qvalue
0   KEGG:05200                                 Pathways in cancer  0.000003
1   KEGG:01521          EGFR tyrosine kinase inhibitor resistance  0.000006
2   KEGG:04010                             MAPK signaling pathway  0.000069
3   KEGG:04151                         PI3K-Akt signaling pathway  0.000094
4   KEGG:04510                                     Focal adhesion  0.005296
5   KEGG:04550  Signaling pathways regulating pluripotency of ...  0.007514
6   KEGG:05215                                    Prostate cancer  0.008197
7   KEGG:04150                             mTOR signaling pathway  0.009719
8   KEGG:05205                            Proteoglycans in cancer  0.009719
9   KEGG:04725                                Cholinergic synapse  0.009719
10  KEGG:05222                             Small cell lung cancer  0.009719
11  KEGG:05214                                             Glioma  0.009719
12  KEGG:040

In [17]:
# Print the membership edges ordered by q-value
if not edges_df.empty and not nodes_df.empty:
    print(
        pd.merge(
            left=edges_df,
            right=nodes_df,
            how='inner',
            left_on='target',
            right_on='id'
        )
        [['source', 'target', 'label', 'qvalue']]
        .sort_values(by='qvalue', ascending=True, ignore_index=True)
    )

     source      target                       label    qvalue
0      ESR1  KEGG:05200          Pathways in cancer  0.000003
1     SMAD2  KEGG:05200          Pathways in cancer  0.000003
2      SKP1  KEGG:05200          Pathways in cancer  0.000003
3    ZBTB16  KEGG:05200          Pathways in cancer  0.000003
4     NCOA3  KEGG:05200          Pathways in cancer  0.000003
..      ...         ...                         ...       ...
377    AKT3  KEGG:04630  JAK-STAT signaling pathway  0.048904
378    SOS1  KEGG:04630  JAK-STAT signaling pathway  0.048904
379   CCND2  KEGG:04630  JAK-STAT signaling pathway  0.048904
380    JAK1  KEGG:04630  JAK-STAT signaling pathway  0.048904
381    BCL2  KEGG:04630  JAK-STAT signaling pathway  0.048904

[382 rows x 4 columns]


### Normal

In [18]:
# Create the gene-pathway membership network for the group
edges_df, nodes_df = create_gene_pathway_network('Normal')

In [19]:
# Print all enriched pathways for the group
if not edges_df.empty and not nodes_df.empty:
    print(
        nodes_df
        .query('type == "Pathway"')
        .reset_index(drop=True)
        [['id', 'label', 'qvalue']]
    )

             id                                              label  \
0    KEGG:04360                                      Axon guidance   
1    KEGG:05205                            Proteoglycans in cancer   
2    KEGG:04517                                 IgSF CAM signaling   
3    KEGG:04010                             MAPK signaling pathway   
4    KEGG:04550  Signaling pathways regulating pluripotency of ...   
..          ...                                                ...   
106  KEGG:04664                    Fc epsilon RI signaling pathway   
107  KEGG:05217                               Basal cell carcinoma   
108  KEGG:05130              Pathogenic Escherichia coli infection   
109  KEGG:05235  PD-L1 expression and PD-1 checkpoint pathway i...   
110  KEGG:04668                              TNF signaling pathway   

           qvalue  
0    4.342067e-07  
1    4.342067e-07  
2    4.416564e-05  
3    4.416564e-05  
4    4.416564e-05  
..            ...  
106  4.632242e-02  

In [20]:
# Print the membership edges ordered by q-value
if not edges_df.empty and not nodes_df.empty:
    print(
        pd.merge(
            left=edges_df,
            right=nodes_df,
            how='inner',
            left_on='target',
            right_on='id'
        )
        [['source', 'target', 'label', 'qvalue']]
        .sort_values(by='qvalue', ascending=True, ignore_index=True)
    )

        source      target                    label        qvalue
0         NRP1  KEGG:04360            Axon guidance  4.342067e-07
1         SOS2  KEGG:05205  Proteoglycans in cancer  4.342067e-07
2        ACTG1  KEGG:05205  Proteoglycans in cancer  4.342067e-07
3        ERBB4  KEGG:05205  Proteoglycans in cancer  4.342067e-07
4     ARHGEF12  KEGG:05205  Proteoglycans in cancer  4.342067e-07
...        ...         ...                      ...           ...
3757     MAPK1  KEGG:04668    TNF signaling pathway  4.986978e-02
3758     MAPK8  KEGG:04668    TNF signaling pathway  4.986978e-02
3759      CSF1  KEGG:04668    TNF signaling pathway  4.986978e-02
3760      CYLD  KEGG:04668    TNF signaling pathway  4.986978e-02
3761      BCL3  KEGG:04668    TNF signaling pathway  4.986978e-02

[3762 rows x 4 columns]


# Tripartite Network

## Function

In [21]:
def create_tripartite_network(group):
    group_dir = (group.lower()).replace(' ', '-')
    dir_path = NETWORK_DATA_DIRS['processed'][group_dir]

    # Create DataFrames to represent the microRNA-mRNA network
    int_edges_file = NETWORK_FILES['interaction-edges']
    int_edges_df = (
        pd.read_csv(os.path.join(dir_path, int_edges_file))
    )
    int_nodes_file = (NETWORK_FILES['interaction-nodes'])
    int_nodes_df = (
        pd.read_csv(os.path.join(dir_path, int_nodes_file))
    )

    # Create DataFrames to represent the mRNA-pathway network
    mem_edges_file = 'membership-network-edges.csv'
    mem_edges_df = (
        pd.read_csv(os.path.join(dir_path, mem_edges_file))
    )

    mem_nodes_file = 'membership-network-nodes.csv'
    mem_nodes_df = (
        pd.read_csv(os.path.join(dir_path, mem_nodes_file))
    )

    # Create DataFrames to represent the microRNA-mRNA-pathway network
    nodes_df = pd.concat(
        objs=[int_nodes_df, mem_nodes_df],
        ignore_index=True,
    )

    edges_df = pd.concat(
        objs=[int_edges_df, mem_edges_df],
        ignore_index=True,
    )

    # Keep the tripartite network empty if there is no membership network
    if mem_edges_df.empty or mem_nodes_df.empty:
        nodes_df = nodes_df.iloc[0:0]
        edges_df = edges_df.iloc[0:0]
    else:
        nodes_df = nodes_df.drop_duplicates(
            subset='id',
            ignore_index=True,
        )

    # Store the DataFrame of edges
    file_name = 'tripartite-network-edges.csv'
    file_path = os.path.join(dir_path, file_name)
    edges_df.to_csv(file_path, index=False)

    # Store the DataFrame of nodes
    file_name = 'tripartite-network-nodes.csv'
    file_path = os.path.join(dir_path, file_name)
    nodes_df.to_csv(file_path, index=False)

    return edges_df, nodes_df

## Construction of Network by Group

### Basal-like

In [22]:
# Create the miR-mRNA-pathway network for the group
edges_df, nodes_df = create_tripartite_network('Basal-like')

In [23]:
# Print the list of edges of the tripartite network
edges_df

,source,target,mirtarbase,correlation,qvalue
0,hsa-miR-182-5p,RAB27A,NaN,-0.564,0.000
1,hsa-miR-18a-5p,ORAI3,MIRT050737,-0.545,0.000
2,hsa-miR-18a-5p,ZBTB4,NaN,-0.532,0.001
3,hsa-miR-96-5p,RAB27A,NaN,-0.496,0.004
4,hsa-miR-106b-5p,ATXN1,MIRT441140,-0.495,0.004
...,...,...,...,...,...
383,ESR1,KEGG:05205,NaN,NaN,NaN
384,TLR4,KEGG:05205,NaN,NaN,NaN
385,IGF1R,KEGG:05205,NaN,NaN,NaN
386,PRKACB,KEGG:05205,NaN,NaN,NaN


In [24]:
# Print the attributes of the tripartite network nodes
nodes_df

,id,label,type,qvalue
0,hsa-miR-182-5p,hsa-miR-182-5p,MicroRNA,NaN
1,hsa-miR-18a-5p,hsa-miR-18a-5p,MicroRNA,NaN
2,hsa-miR-96-5p,hsa-miR-96-5p,MicroRNA,NaN
3,hsa-miR-106b-5p,hsa-miR-106b-5p,MicroRNA,NaN
4,hsa-miR-221-3p,hsa-miR-221-3p,MicroRNA,NaN
...,...,...,...,...
295,KEGG:01521,EGFR tyrosine kinase inhibitor resistance,Pathway,0.025434
296,KEGG:01522,Endocrine resistance,Pathway,0.042191
297,KEGG:04010,MAPK signaling pathway,Pathway,0.042191
298,KEGG:04068,FoxO signaling pathway,Pathway,0.042191


### HER2-enriched

In [25]:
# Create the miR-mRNA-pathway network for the group
edges_df, nodes_df = create_tripartite_network('HER2-enriched')

In [26]:
# Print the list of edges of the tripartite network
edges_df

,source,target,mirtarbase,correlation,qvalue


In [27]:
# Print the attributes of the tripartite network nodes
nodes_df

,id,label,type,qvalue


### Luminal A

In [28]:
# Create the miR-mRNA-pathway network for the group
edges_df, nodes_df = create_tripartite_network('Luminal A')

In [29]:
# Print the list of edges of the tripartite network
edges_df

,source,target,mirtarbase,correlation,qvalue
0,hsa-miR-200a-3p,WWTR1,NaN,-0.477,0.0
1,hsa-miR-141-3p,PHYHIPL,NaN,-0.452,0.0
2,hsa-miR-96-5p,SGMS2,NaN,-0.438,0.0
3,hsa-miR-29b-3p,PCSK5,NaN,-0.425,0.0
4,hsa-miR-342-3p,FRMD4A,NaN,-0.425,0.0
...,...,...,...,...,...
321,ITGA9,KEGG:05165,NaN,NaN,NaN
322,PPP2CB,KEGG:05165,NaN,NaN,NaN
323,HEY2,KEGG:05165,NaN,NaN,NaN
324,CDK6,KEGG:05165,NaN,NaN,NaN


In [30]:
# Print the attributes of the tripartite network nodes
nodes_df

,id,label,type,qvalue
0,hsa-miR-200a-3p,hsa-miR-200a-3p,MicroRNA,NaN
1,hsa-miR-141-3p,hsa-miR-141-3p,MicroRNA,NaN
2,hsa-miR-96-5p,hsa-miR-96-5p,MicroRNA,NaN
3,hsa-miR-29b-3p,hsa-miR-29b-3p,MicroRNA,NaN
4,hsa-miR-342-3p,hsa-miR-342-3p,MicroRNA,NaN
...,...,...,...,...
253,KEGG:04518,Integrin signaling,Pathway,0.018969
254,KEGG:04820,Cytoskeleton in muscle cells,Pathway,0.018969
255,KEGG:05202,Transcriptional misregulation in cancer,Pathway,0.018969
256,KEGG:04151,PI3K-Akt signaling pathway,Pathway,0.024587


### Luminal B

In [31]:
# Create the miR-mRNA-pathway network for the group
edges_df, nodes_df = create_tripartite_network('Luminal B')

In [32]:
# Print the list of edges of the tripartite network
edges_df

,source,target,mirtarbase,correlation,qvalue
0,hsa-miR-200c-3p,ZMAT3,MIRT209182,-0.493,0.000
1,hsa-let-7g-5p,KLHL13,NaN,-0.467,0.001
2,hsa-miR-30d-5p,FST,NaN,-0.461,0.001
3,hsa-miR-200c-3p,FAM126A,NaN,-0.458,0.001
4,hsa-miR-151a-3p,FRK,MIRT508167,-0.456,0.001
...,...,...,...,...,...
961,SOS1,KEGG:04630,NaN,NaN,NaN
962,CCND2,KEGG:04630,NaN,NaN,NaN
963,JAK1,KEGG:04630,NaN,NaN,NaN
964,IL6ST,KEGG:04630,NaN,NaN,NaN


In [33]:
# Print the attributes of the tripartite network nodes
nodes_df

,id,label,type,qvalue
0,hsa-miR-200c-3p,hsa-miR-200c-3p,MicroRNA,NaN
1,hsa-let-7g-5p,hsa-let-7g-5p,MicroRNA,NaN
2,hsa-miR-30d-5p,hsa-miR-30d-5p,MicroRNA,NaN
3,hsa-miR-151a-3p,hsa-miR-151a-3p,MicroRNA,NaN
4,hsa-miR-27b-3p,hsa-miR-27b-3p,MicroRNA,NaN
...,...,...,...,...
585,KEGG:04926,Relaxin signaling pathway,Pathway,0.038553
586,KEGG:04012,ErbB signaling pathway,Pathway,0.038965
587,KEGG:04068,FoxO signaling pathway,Pathway,0.041710
588,KEGG:04720,Long-term potentiation,Pathway,0.045329


### Normal

In [34]:
# Create the miR-mRNA-pathway network for the group
edges_df, nodes_df = create_tripartite_network('Normal')

In [35]:
# Print the list of edges of the tripartite network
edges_df

,source,target,mirtarbase,correlation,qvalue
0,hsa-miR-22-3p,RPS3,NaN,-0.816,0.0
1,hsa-miR-452-5p,RPA1,NaN,-0.810,0.0
2,hsa-miR-141-3p,TRHDE,NaN,-0.807,0.0
3,hsa-miR-22-3p,RPL14,NaN,-0.804,0.0
4,hsa-miR-22-3p,LRRC1,MIRT030660,-0.800,0.0
...,...,...,...,...,...
8829,FAS,KEGG:04668,NaN,NaN,NaN
8830,CASP10,KEGG:04668,NaN,NaN,NaN
8831,LIF,KEGG:04668,NaN,NaN,NaN
8832,CSF1,KEGG:04668,NaN,NaN,NaN


In [36]:
# Print the attributes of the tripartite network nodes
nodes_df

,id,label,type,qvalue
0,hsa-miR-22-3p,hsa-miR-22-3p,MicroRNA,NaN
1,hsa-miR-452-5p,hsa-miR-452-5p,MicroRNA,NaN
2,hsa-miR-141-3p,hsa-miR-141-3p,MicroRNA,NaN
3,hsa-miR-200a-3p,hsa-miR-200a-3p,MicroRNA,NaN
4,hsa-miR-224-5p,hsa-miR-224-5p,MicroRNA,NaN
...,...,...,...,...
3021,KEGG:04664,Fc epsilon RI signaling pathway,Pathway,0.046322
3022,KEGG:05217,Basal cell carcinoma,Pathway,0.047513
3023,KEGG:05130,Pathogenic Escherichia coli infection,Pathway,0.047713
3024,KEGG:05235,PD-L1 expression and PD-1 checkpoint pathway i...,Pathway,0.048613
